# 差区间归因与条件信息SOP
预先固定四个状态：5/20日方向组合。完整保留所有季度和状态，不用未来收益定义牛熊，不把归因当作删掉现金后的收益。先诊断并检查现金评分条件信息，再决定是否有新增拟合依据。

In [1]:
from pathlib import Path
import os,json,hashlib
import pandas as pd
import qlib
from qlib.workflow import R
from src.analysis.regime_failure import regime_features,account_groups,conditional_information
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
card=root/'research/regime_failure_sop_20261014';plan=json.loads((card/'plan.json').read_text())
provider=root/'data/qlib/full_path_tiered_allocation_20261013';qlib.init(provider_uri=str(provider),region='us')
def read(source,name):
    spec=plan['sources'][source];path=root/spec['card'];manifest=path/'verification_manifest.json'
    assert hashlib.sha256(manifest.read_bytes()).hexdigest()==spec['manifest_sha256']
    entries=json.loads(manifest.read_text())['entries'];entry=next(e for e in entries if e['path'].endswith('/artifacts/'+name))
    with R.uri_context(str(path/'qlib_recorder')):
        rec=R.get_recorder(recorder_id=spec['recorder_id'],experiment_name=source)
        artifact=Path(rec.download_artifact(name));assert hashlib.sha256(artifact.read_bytes()).hexdigest()==entry['sha256']
        return rec.load_object(name)
frame=read('full_path_tiered_allocation_20261013','inputs/frame');states=regime_features(frame)
labels=read('allocator_cash_information_20261014','analysis/daily')
information=conditional_information(labels,states,repetitions=plan['bootstrap']['repetitions'],seed=plan['bootstrap']['seed'])
quarters=[];regimes=[];joined={}
for slip in plan['costs']:
    daily=read('anchor_decision_attribution_20261014','analysis/daily/current_anchor_'+str(slip))
    joined[str(slip)],tables=account_groups(daily,states)
    for name,collection in [('quarter',quarters),('regime',regimes)]:
        table=tables[name];table['slippage_bps']=slip;collection.append(table)
quarter=pd.concat(quarters,ignore_index=True);regime=pd.concat(regimes,ignore_index=True)
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='fixed_regime_diagnosis',uri=str(tracking)):
    rec=R.get_recorder();rec.log_params(new_fits=0,new_backtests=0,regime_count=4)
    rec.save_objects(artifact_path='inputs',plan=plan,regime_inputs=states)
    rec.save_objects(artifact_path='analysis',daily=joined,quarter=quarter,regime=regime,conditional_information=information)
    rec.save_objects(artifact_path='sources',analysis=(root/'src/analysis/regime_failure.py').read_text())
    artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
summary={'experiment':card.name,'decision':'diagnostic_only_conditional_information_gate','objective_achieved':False,'new_fits':0,'new_backtests':0,'regime_count':4,'account_sessions':len(joined['2']),'positive_lower_bound_cells':int(information.lower.gt(0).sum()),'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(tracking),'provider_uri':str(provider),'region':'us','as_of':plan['as_of'],'qlib_docs':'https://qlib.readthedocs.io/en/latest/component/recorder.html','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/daily','sources/*']}}
atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json')
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json')
quarter.to_csv(card/'quarter_attribution.csv',index=False);regime.to_csv(card/'regime_attribution.csv',index=False);information.to_csv(card/'conditional_information.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']},sort_keys=False))
print(quarter.loc[quarter.slippage_bps.eq(2)].sort_values('relative_log').head(4).to_string(index=False));print(regime.to_string(index=False));print(information.to_string(index=False))


# 差区间原因与条件信息诊断

复用已校验Qlib原生完整账户，944区间，10万美元、IBKR Pro Tiered，2/5bp独立路径；0拟合、0新增回测。四状态仅用决策前信号日的SOXX 5/20日收益符号；所有季度/状态均报告，不能把事后最差窗口当作训练选择。

2bp最差四季度：2023Q4相对财富变化-12.55%，2026Q3 -10.86%，2023Q2 -8.81%，2025Q4 -8.28%。相对财富变化是策略期末增长倍数除以SOXX增长倍数减1，不是收益百分点。

2023Q4相对对数贡献-0.1341，其中现金两部分合计-0.1065、SOXL -0.0262；2026Q3合计-0.1150，其中现金-0.0394、SOXL -0.0749。两个季度的主要错误不同，不能都归为费用。

预先定义的过去5/20日均下跌状态共204日，2bp实际平均现金61.77%，相对对数贡献-0.4730，现金合计-0.4770；该组足以抵消其他状态中的正贡献。它指过去下跌，不指未来仍下跌。旧模型原始现金在该状态均值42.61%，与实际账户现金不同；两者差距含完整控制器与持仓漂移，不能直接视为可移除的无效风控。5bp该组现金66.84%，相对对数贡献-0.7513。

2bp上行状态贡献+0.2270，但5bp转为-0.2991；既有阈值分歧证据提示账户路径放大，不是本卡证明的单一因果。

对旧单模型和base ensemble，在四状态分别检验H5现金优势排序，共8项95%区块区间全部跨0。旧模型下跌状态IC=0.1681、区间[-0.0230,0.3459]，其余状态点估计为负；不能据此分状态翻号或切换模型。区间保留63日相关结构，1000次重采样，未校正此前多轮研究选择。

结论：主要问题集中于下跌后的持仓与修复识别，以及部分季度SOXL相对选择；费用不是唯一原因。归因是会计恒等式，不能证明少持现金就能在风险约束下收回损失。诊断后已启动并完成独立瘦卡中的两特征固定试验，结果未通过，见相邻drawdown_transition_information_20261014。

决定diagnostic_only_no_promotion。稳定超额未实现；现役生产未改。下一步按[研究SOP](../../docs/research_failure_to_feature_sop.md)核查成分内部修复广度的信息资格。

Recorder `8220d22658b044e0b6586aba722618ee`。
